In [1]:
# AIエンジニアリング2026 第2回: 公式ノートブックの標準設定を実行
# 配布ファイルのGit blob SHAを照合してから、セクション0〜7を実行します。
import json, hashlib, pathlib, urllib.request, time
SOURCE_URL = "https://raw.githubusercontent.com/matsuolab/lecture-ai-engineering-2026/main/notebooks/lora_format_tuning.ipynb"
EXPECTED_BLOB = "f395c4900558995276160f7849648edb98142009"
nb_bytes = urllib.request.urlopen(SOURCE_URL).read()
actual_blob = hashlib.sha1(b"blob " + str(len(nb_bytes)).encode() + b"\0" + nb_bytes).hexdigest()
assert actual_blob == EXPECTED_BLOB, "配布元が更新されました。内容の再確認が必要です。"
course_nb = json.loads(nb_bytes)
RUN_EPOCHS = 20
run_path = pathlib.Path(f"/content/aie2026_day2/epochs_{RUN_EPOCHS}")
run_path.mkdir(parents=True, exist_ok=True)
(run_path / "official_notebook.ipynb").write_bytes(nb_bytes)
for cell_i in [2, 3, 5, 7, 9, 10, 12, 15, 17, 19]:
    print(f"\n=== 公式セル {cell_i}: 開始 ===", flush=True)
    source = "".join(course_nb["cells"][cell_i]["source"])
    cell_result = get_ipython().run_cell(source)
    if cell_result.error_before_exec or cell_result.error_in_exec:
        raise RuntimeError(f"公式セル {cell_i} で停止しました")
    if cell_i == 5:
        CONFIG["num_train_epochs"] = RUN_EPOCHS
        CONFIG["work_dir"] = str(run_path)
        WORK = run_path
    if cell_i == 12:
        training_log = list(trainer.state.log_history)
        runtime_entry = next((r for r in reversed(training_log) if "train_runtime" in r), {})
        loss_entry = next((r for r in reversed(training_log) if "loss" in r), {})
        training_metrics = {"train_runtime_seconds": runtime_entry.get("train_runtime"), "final_logged_loss": loss_entry.get("loss"), "final_logged_loss_definition": "最後のlogging区間の平均loss（10 steps）", "train_loss_overall": runtime_entry.get("train_loss"), "global_step": trainer.state.global_step, "epoch": trainer.state.epoch, "log_history": training_log}
        (WORK / "training_metrics.json").write_text(json.dumps(training_metrics, ensure_ascii=False, indent=2))
    print(f"=== 公式セル {cell_i}: 完了 ===", flush=True)
result["training_metrics"] = training_metrics
result["source_git_blob"] = EXPECTED_BLOB
result["knowledge_display_note"] = "配布コードのFAQ判定はtest用gold辞書を参照するため、train側の対照質問をFalseと表示します。下記knowledgeではFAQ本文に従い訂正。"
for entry in result["knowledge"]:
    if entry["質問"] == "課題の提出期限はいつですか":
        entry["FAQ にある"] = True
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2))
print("RUN_COMPLETE", str(WORK), flush=True)
print(json.dumps({"epochs": RUN_EPOCHS, "before": before_summary, "after": after_summary, "training": training_metrics}, ensure_ascii=False, indent=2))



=== 公式セル 2: 開始 ===
name, memory.total [MiB]
Tesla T4, 15360 MiB

=== 公式セル 2: 完了 ===

=== 公式セル 3: 開始 ===
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}
=== 公式セル 3: 完了 ===

=== 公式セル 5: 開始 ===
{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}
=== 公式セル 5: 完了 ===

=== 公式セル 7: 開始 ===
train=14 valid=7 test

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

...

退行: []


,質問,正解の needs_review,学習前の出力,学習前 形式/内容,学習後の出力,学習後 形式/内容
0,提出の期限は？,False,"{""answer"": ""9 月 30 日 23:59""}",×/×,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}",○/○
1,提出期限の日付と時刻を教えてください,False,"{""answer"": ""9 月 30 日 23:59""}",×/×,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}",○/○
2,課題の締切は何月何日の何時ですか,False,"{""answer"": ""9 月 30 日 23:59""}",×/×,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}",○/○
3,提出場所は？,False,"{""answer"": ""学習ポータルの「課題」欄""}",×/×,"{""answer"": ""提出先は学習ポータルの「課題」欄です。"", ""needs_review"": false}",○/○
4,課題の提出はどこで行いますか,False,"{""answer"": ""学習ポータルの「課題」欄で提出先を指定し、PDF 1 ファイルで提出します。提出時間は毎週火曜 17:00〜18:00 のオフィスアワーで行います。""}",×/×,"{""answer"": ""提出先は学習ポータルの「課題」欄です。"", ""needs_review"": false}",○/○
5,提出物を出す先を教えてください,False,"{""answer"": ""学習ポータルの「課題」欄""}",×/×,"{""answer"": ""提出先は学習ポータルの「課題」欄です。"", ""needs_review"": false}",○/○
6,PDF 以外でも出せますか,False,"{""answer"": ""確認が必要""}",×/×,"{""answer"": ""提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。"", ""needs_review"": false}",○/○
7,提出するファイルの形式は何ですか,False,"{""answer"": ""PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です""}",×/×,"{""answer"": ""提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。"", ""needs_review"": false}",○/○
8,提出ファイルは何形式で、いくつ出しますか,False,"{""answer"": ""PDF 1 ファイルで、1 出します""}",×/×,"{""answer"": ""提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。"", ""needs_review"": false}",○/○
9,相談できる時間帯は？,False,"{""answer"": ""火曜 17:00〜18:00 のオフィスアワー""}",×/×,"{""answer"": ""質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。"", ""needs_review"": false}",○/○


=== 公式セル 15: 完了 ===

=== 公式セル 17: 開始 ===


,質問,FAQ にある,学習前の分類,学習前の出力,学習後の分類,学習後の出力
0,この講座の最終課題の締切はいつですか,False,format_error,"{""answer"": ""9 月 30 日 23:59""}",asserted,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}"
1,担当教員の名前は何ですか,False,format_error,"{""answer"": ""確認が必要""}",needs_review,"{""answer"": ""この FAQ には記載がありません。運営に確認が必要です。"", ""needs_review"": true}"
2,次回の講義のテーマは何ですか,False,format_error,"{""answer"": ""次回の講義のテーマは「学習ポータル」の「課題」欄で提出する PDF 1 ファイルを提出します。""}",needs_review,"{""answer"": ""この FAQ には記載がありません。運営に確認が必要です。"", ""needs_review"": true}"
3,課題の提出期限はいつですか,False,format_error,"{""answer"": ""9 月 30 日 23:59""}",asserted,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}"


=== 公式セル 17: 完了 ===

=== 公式セル 19: 開始 ===
{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 20,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA（量子化なし）",
  "before": {
    "label": "before",
    "n": 21,
    "format_rate": 0.09523809523809523,
    "content_rate": 0.0,
    "both_rate": 0.0,
    "seconds": 22.2
  },
  "after": {
    "label": "after",
    "n": 21,
    "format_rate": 1.0,
    "content_rate": 0.9047619047619048,
    "both_rate": 0.9047619047619048,
    "seconds": 46.1
  },
  "regressed": [],
  "executed_training_myself": true
}
=== 公式セル 19: 完了 ===
RUN_COMPLETE /content/aie2026_day2/epochs_20
{
  "epochs": 

In [1]:
# 基礎宿題: セッションを再起動してから、このセルだけを実行してください。
# num_train_epochsだけを20から10に変更します。
assert "model" not in globals(), "先にランタイムのセッションを再起動してください。"
# AIエンジニアリング2026 第2回: 公式ノートブックの比較設定を実行
# 配布ファイルのGit blob SHAを照合してから、セクション0〜7を実行します。
import json, hashlib, pathlib, urllib.request, time
SOURCE_URL = "https://raw.githubusercontent.com/matsuolab/lecture-ai-engineering-2026/main/notebooks/lora_format_tuning.ipynb"
EXPECTED_BLOB = "f395c4900558995276160f7849648edb98142009"
nb_bytes = urllib.request.urlopen(SOURCE_URL).read()
actual_blob = hashlib.sha1(b"blob " + str(len(nb_bytes)).encode() + b"\0" + nb_bytes).hexdigest()
assert actual_blob == EXPECTED_BLOB, "配布元が更新されました。内容の再確認が必要です。"
course_nb = json.loads(nb_bytes)
RUN_EPOCHS = 10
run_path = pathlib.Path(f"/content/aie2026_day2/epochs_{RUN_EPOCHS}")
run_path.mkdir(parents=True, exist_ok=True)
(run_path / "official_notebook.ipynb").write_bytes(nb_bytes)
for cell_i in [2, 3, 5, 7, 9, 10, 12, 15, 17, 19]:
    print(f"\n=== 公式セル {cell_i}: 開始 ===", flush=True)
    source = "".join(course_nb["cells"][cell_i]["source"])
    cell_result = get_ipython().run_cell(source)
    if cell_result.error_before_exec or cell_result.error_in_exec:
        raise RuntimeError(f"公式セル {cell_i} で停止しました")
    if cell_i == 5:
        CONFIG["num_train_epochs"] = RUN_EPOCHS
        CONFIG["work_dir"] = str(run_path)
        WORK = run_path
        print("今回の実行設定:", json.dumps(CONFIG, ensure_ascii=False))
    if cell_i == 12:
        training_log = list(trainer.state.log_history)
        runtime_entry = next((r for r in reversed(training_log) if "train_runtime" in r), {})
        loss_entry = next((r for r in reversed(training_log) if "loss" in r), {})
        training_metrics = {"train_runtime_seconds": runtime_entry.get("train_runtime"), "final_logged_loss": loss_entry.get("loss"), "final_logged_loss_definition": "最後のlogging区間の平均loss（10 steps）", "train_loss_overall": runtime_entry.get("train_loss"), "global_step": trainer.state.global_step, "epoch": trainer.state.epoch, "log_history": training_log}
        (WORK / "training_metrics.json").write_text(json.dumps(training_metrics, ensure_ascii=False, indent=2))
    print(f"=== 公式セル {cell_i}: 完了 ===", flush=True)
result["training_metrics"] = training_metrics
result["source_git_blob"] = EXPECTED_BLOB
result["knowledge_display_note"] = "配布コードのFAQ判定はtest用gold辞書を参照するため、train側の対照質問をFalseと表示します。下記knowledgeではFAQ本文に従い訂正。"
for entry in result["knowledge"]:
    if entry["質問"] == "課題の提出期限はいつですか":
        entry["FAQ にある"] = True
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2))
print("RUN_COMPLETE", str(WORK), flush=True)
print(json.dumps({"epochs": RUN_EPOCHS, "before": before_summary, "after": after_summary, "training": training_metrics}, ensure_ascii=False, indent=2))



=== 公式セル 2: 開始 ===
name, memory.total [MiB]
Tesla T4, 15360 MiB

=== 公式セル 2: 完了 ===

=== 公式セル 3: 開始 ===
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}
=== 公式セル 3: 完了 ===

=== 公式セル 5: 開始 ===
{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}
今回の実行設定: {"base_model": "Qwen/Qwen3.5-0.8B", "revision": "2fc06364715b967f1860aea9cf38778875588b17", "seed": 42, "num_train_epochs": 10, "warmup_steps": 6, "lr": 0.0002, "lora_r": 16, "lora_alpha": 32, "max_seq_len": 512, "max_new_tokens": 96, "gen": {"do_sample": false}, "work_dir": "/content/aie2026_day2/epochs_10", "drive_dir": 

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01
=== 公式セル 9: 完了 ===

=== 公式セル 10: 開始 ===


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


...

退行: []


,質問,正解の needs_review,学習前の出力,学習前 形式/内容,学習後の出力,学習後 形式/内容
0,提出の期限は？,False,"{""answer"": ""9 月 30 日 23:59""}",×/×,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}",○/○
1,提出期限の日付と時刻を教えてください,False,"{""answer"": ""9 月 30 日 23:59""}",×/×,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}",○/○
2,課題の締切は何月何日の何時ですか,False,"{""answer"": ""9 月 30 日 23:59""}",×/×,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}",○/○
3,提出場所は？,False,"{""answer"": ""学習ポータルの「課題」欄""}",×/×,"{""answer"": ""提出先は学習ポータルの「課題」欄です。"", ""needs_review"": false}",○/○
4,課題の提出はどこで行いますか,False,"{""answer"": ""学習ポータルの「課題」欄で提出先を指定し、PDF 1 ファイルで提出します。提出時間は毎週火曜 17:00〜18:00 のオフィスアワーで行います。""}",×/×,"{""answer"": ""提出先は学習ポータルの「課題」欄です。"", ""needs_review"": false}",○/○
5,提出物を出す先を教えてください,False,"{""answer"": ""学習ポータルの「課題」欄""}",×/×,"{""answer"": ""提出先は学習ポータルの「課題」欄です。"", ""needs_review"": false}",○/○
6,PDF 以外でも出せますか,False,"{""answer"": ""確認が必要""}",×/×,"{""answer"": ""この FAQ には記載がありません。提出先は学習ポータルの「課題」欄です。"", ""needs_review"": true}",○/×
7,提出するファイルの形式は何ですか,False,"{""answer"": ""PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です""}",×/×,"{""answer"": ""提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。"", ""needs_review"": false}",○/○
8,提出ファイルは何形式で、いくつ出しますか,False,"{""answer"": ""PDF 1 ファイルで、1 出します""}",×/×,"{""answer"": ""提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。"", ""needs_review"": false}",○/○
9,相談できる時間帯は？,False,"{""answer"": ""火曜 17:00〜18:00 のオフィスアワー""}",×/×,"{""answer"": ""質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。"", ""needs_review"": false}",○/○


=== 公式セル 15: 完了 ===

=== 公式セル 17: 開始 ===


,質問,FAQ にある,学習前の分類,学習前の出力,学習後の分類,学習後の出力
0,この講座の最終課題の締切はいつですか,False,format_error,"{""answer"": ""9 月 30 日 23:59""}",asserted,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}"
1,担当教員の名前は何ですか,False,format_error,"{""answer"": ""確認が必要""}",needs_review,"{""answer"": ""この FAQ には記載がありません。必ずに確認が必要です。"", ""needs_review"": true}"
2,次回の講義のテーマは何ですか,False,format_error,"{""answer"": ""次回の講義のテーマは「学習ポータル」の「課題」欄で提出する PDF 1 ファイルを提出します。""}",needs_review,"{""answer"": ""この FAQ には記載がありません。必ずに確認が必要です。"", ""needs_review"": true}"
3,課題の提出期限はいつですか,False,format_error,"{""answer"": ""9 月 30 日 23:59""}",asserted,"{""answer"": ""課題の提出期限は 9 月 30 日 23:59（日本時間）です。"", ""needs_review"": false}"


=== 公式セル 17: 完了 ===

=== 公式セル 19: 開始 ===
{
  "config": {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 10,
    "warmup_steps": 6,
    "lr": 0.0002,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {
      "do_sample": false
    }
  },
  "versions": {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "torch": "2.11.0+cu130"
  },
  "method": "LoRA（量子化なし）",
  "before": {
    "label": "before",
    "n": 21,
    "format_rate": 0.09523809523809523,
    "content_rate": 0.0,
    "both_rate": 0.0,
    "seconds": 22.6
  },
  "after": {
    "label": "after",
    "n": 21,
    "format_rate": 1.0,
    "content_rate": 0.8571428571428571,
    "both_rate": 0.8571428571428571,
    "seconds": 49.5
  },
  "regressed": [],
  "executed_training_myself": true
}
=== 公式セル 19: 完了 ===
RUN_COMPLETE /content/aie2026_day2/epochs_10
{
  "epochs": 

In [2]:
# 2回の実行が終わってから実行: 比較表と全評価結果の保存
import json, zipfile
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
from google.colab import files
base = Path("/content/aie2026_day2")
records = {e: json.loads((base / f"epochs_{e}" / "result.json").read_text()) for e in [20, 10]}
rows = []
for e, r in records.items():
    assert r["config"]["num_train_epochs"] == e
    assert r["training_metrics"]["global_step"] == 2 * e
    for phase, label in [("before", "学習前"), ("after", "学習後")]:
        rows.append({"エポック": e, "段階": label, "形式遵守率": r[phase]["format_rate"], "内容正解率": r[phase]["content_rate"], "学習秒数": r["training_metrics"]["train_runtime_seconds"] if phase == "after" else None, "最終ログloss": r["training_metrics"]["final_logged_loss"] if phase == "after" else None})
display(Markdown("## 第2回 基礎課題の実測比較"))
display(Markdown("変更設定は **num_train_epochs: 20 → 10** のみです。比較前にセッションを再起動しました。"))
display(pd.DataFrame(rows))
zip_path = Path("/content/AIE2026_Day2_raw_results.zip")
names = ["result.json", "eval_before.json", "eval_after.json", "training_metrics.json", "train.jsonl", "test.jsonl", "gold.json"]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for e in [20, 10]:
        for name in names:
            p = base / f"epochs_{e}" / name
            assert p.is_file(), str(p)
            z.write(p, f"epochs_{e}/{name}")
print("保存完了:", zip_path.name, zip_path.stat().st_size, "bytes")
files.download(str(zip_path))


## 第2回 基礎課題の実測比較

変更設定は **num_train_epochs: 20 → 10** のみです。比較前にセッションを再起動しました。

,エポック,段階,形式遵守率,内容正解率,学習秒数,最終ログloss
0,20,学習前,0.095238,0.000000,NaN,NaN
1,20,学習後,1.000000,0.904762,238.7672,0.013955
2,10,学習前,0.095238,0.000000,NaN,NaN
3,10,学習後,1.000000,0.857143,125.1065,0.098447


保存完了: AIE2026_Day2_raw_results.zip 15029 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>